In [9]:
%pip install torch numpy matplotlib torchvision pandas scikit-learn opencv-python librosa

Note: you may need to restart the kernel to use updated packages.


In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
import numpy as np
import cv2 as cv
import pandas as p
import glob

In [11]:
# Defining the Simple CNN architecture with 
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
    
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [12]:
# Loading the ResNet18 architecture from torchvision
from torchvision import models
resnet18 = models.resnet18()

# Loading the ResNet34 architecture from torchvision
resnet34 = models.resnet34()

In [13]:
# Loading the five datasets from FakeMusicCaps
audio_ldm = glob.glob('FakeMusicCaps/audioldm2/*.wav')
music_gen = glob.glob('FakeMusicCaps/MusicGen_medium/*.wav')
music_ldm = glob.glob('FakeMusicCaps/musicldm/*.wav')
mus_tango = glob.glob('FakeMusicCaps/mustango/*.wav')
stable_audio_open = glob.glob('FakeMusicCaps/stable_audio_open/*.wav')

# Store the datasets in a dataloader
class AudioDataset(Dataset):
    def __init__(self, file_paths, transform=None):
        self.file_paths = file_paths
        self.transform = transform

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        audio_path = self.file_paths[idx]
        audio_data, sample_rate = librosa.load(audio_path, sr=None)
        if self.transform:
            audio_data = self.transform(audio_data)
        return audio_data


In [14]:
# Train the model on an audio dataset
def train_model(model, dataloader, criterion, optimizer, num_epochs=25):
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in dataloader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
        epoch_loss = running_loss / len(dataloader.dataset)
        print(f'Epoch {epoch}/{num_epochs - 1}, Loss: {epoch_loss:.4f}')

In [15]:
# Load the data onto the GPU if available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
# Prepare the CNN model, loss function, and optimizer for training
cnn_model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cnn_model.parameters(), lr=0.001)

# Example of creating a dataloader for one of the datasets
transform = transforms.Compose([transforms.ToTensor()])
audio_dataset = AudioDataset(audio_ldm, transform=transform)
dataloader = DataLoader(audio_dataset, batch_size=32, shuffle=True)

ValueError: num_samples should be a positive integer value, but got num_samples=0

In [ ]:
# Move the model to the GPU
model = SimpleCNN().to(device)

In [ ]:
# Train the model
train_model(cnn_model, dataloader, criterion, optimizer, num_epochs=25)